# Traffic Flow Prediction - EDA & Modelling
Branch: `Traffic-Flow-Prediction_AI133`  
Dataset: Kaggle *Traffic Prediction Dataset* (DateTime, Junction, Vehicles, ID)

Run this notebook from the `notebooks/` folder.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
import sys; sys.path.append('../src')
from preprocess import clean_data
from features import add_features, get_X_y

raw = pd.read_csv('../data/raw/traffic.csv')
raw.head()

In [ ]:
raw.info()
raw.describe()

In [ ]:
raw.isnull().sum()

In [ ]:
df = clean_data(raw)
df.groupby('Junction').DateTime.agg(['min','max','count'])

## EDA

In [ ]:
df.groupby('Junction').Vehicles.describe()

In [ ]:
df.set_index('DateTime').groupby('Junction').Vehicles.resample('D').mean().unstack(0).plot(figsize=(12,4), title='Daily average vehicles per junction')
plt.show()

In [ ]:
df['hour'] = df.DateTime.dt.hour
sns.lineplot(data=df, x='hour', y='Vehicles', hue='Junction')
plt.title('Average vehicles by hour of day'); plt.show()

In [ ]:
df['dow'] = df.DateTime.dt.dayofweek
sns.barplot(data=df, x='dow', y='Vehicles', hue='Junction')
plt.title('Average vehicles by day of week (0=Mon)'); plt.show()

## Features

In [ ]:
feat = add_features(df.drop(columns=['hour','dow']))
X, y = get_X_y(feat)
X.head()

## Models
Full training + comparison is in `src/train.py`. Quick check here:

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
cut = feat.DateTime.quantile(0.8)
tr = feat.DateTime < cut
rf = RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=42).fit(X[tr], y[tr])
p = rf.predict(X[~tr])
print('MAE', mean_absolute_error(y[~tr], p), 'R2', r2_score(y[~tr], p))